In [1]:
!pip install -q torch \
 transformers \
 bitsandbytes \
 nvidia-ml-py \
 langchain-groq \
 langchain-google-genai \
 langchain-mistralai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.5/70.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 15.0 MB/s eta 0:00:00


In [2]:
!pip install -q --upgrade transformers accelerate tokenizers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 27.4 MB/s eta 0:00:00


In [3]:
import torch
import time
import re
import threading
import pandas as pd
import pynvml
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_mistralai import ChatMistralAI
from langchain.messages import HumanMessage

print("Imports successful")

Imports successful


In [4]:
! nvidia-smi

Tue Jun 30 11:19:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
df = pd.read_json('benchmark.json')
print(df["prompt"][1])

If a clock strikes 6 times in 5 seconds, how many seconds will it take to strike 12 times?


In [6]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)
# ── Load model ───────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained("microsoft/phi-4", use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/phi-4",
    quantization_config=quantization_config,
    device_map='auto'
)

# ── Load model ───────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained("microsoft/phi-4", use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/phi-4",
    quantization_config=quantization_config,
    device_map='auto'
)


In [ ]:
# ── GPU monitor setup ────────────────────────────────────────────────
pynvml.nvmlInit()
handle = pynvml.nvmlDeviceGetHandleByIndex(0)

# ── Helper: poll GPU in background thread during generation ──────────
def poll_gpu(handle, interval, results, stop_event):
    """Polls GPU every `interval` seconds until stop_event is set."""
    util_readings, temp_readings, mem_readings = [], [], []
    while not stop_event.is_set():
        util_readings.append(pynvml.nvmlDeviceGetUtilizationRates(handle).gpu)
        temp_readings.append(pynvml.nvmlDeviceGetTemperature(handle, pynvml.NVML_TEMPERATURE_GPU))
        mem_readings.append(pynvml.nvmlDeviceGetMemoryInfo(handle).used / 1024**2)
        time.sleep(interval)
    results['gpu_usage_pct']   = round(max(util_readings), 2)   # peak usage
    results['gpu_temp_c']      = round(max(temp_readings), 2)   # peak temp
    results['gpu_mem_used_mb'] = round(max(mem_readings), 2)    # peak memory

# ── Strip DeepSeek <think> chain-of-thought block ───────────────────
def strip_think_block(text):
    return re.sub(r'<think>.*?</think>', '', str(text), flags=re.DOTALL).strip()

# ── Main inference loop ──────────────────────────────────────────────
phi4_records = []

for i in range(len(df)):
    messages = [{"role": "user", "content": df["prompt"][i]}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    prompt_tokens = inputs["input_ids"].shape[1]

    # Start background GPU polling
    gpu_results = {}
    stop_event  = threading.Event()
    gpu_thread  = threading.Thread(
        target=poll_gpu,
        args=(handle, 0.5, gpu_results, stop_event)  # poll every 0.5s
    )
    gpu_thread.start()

    # Generation
    start = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=2048,
            pad_token_id=tokenizer.eos_token_id
        )
    latency = time.time() - start

    # Stop GPU polling
    stop_event.set()
    gpu_thread.join()

    # Decode only new tokens (strips prompt echo)
    new_tokens    = outputs[0][prompt_tokens:]
    output_tokens = new_tokens.shape[0]
    text          = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    text          = strip_think_block(text)

    phi4_records.append({
        "question_id"     : i,
        "question"        : df["prompt"][i],
        "answer"          : text,
        "prompt_tokens"   : prompt_tokens,
        "output_tokens"   : output_tokens,
        "total_tokens"    : prompt_tokens + output_tokens,
        "latency_sec"     : round(latency, 4),
        "tokens_per_sec"  : round(output_tokens / latency, 2),
        "gpu_usage_pct"   : gpu_results.get('gpu_usage_pct'),
        "gpu_temp_c"      : gpu_results.get('gpu_temp_c'),
        "gpu_mem_used_mb" : gpu_results.get('gpu_mem_used_mb'),
    })


    print(f"[{i+1}/{len(df)}] done | "
          f"out_tokens: {output_tokens} | "
          f"latency: {latency:.2f}s | "
          f"tok/s: {output_tokens/latency:.1f} | "
          f"gpu: {gpu_results.get('gpu_usage_pct')}% | "
          f"temp: {gpu_results.get('gpu_temp_c')}C | "
          f"mem: {gpu_results.get('gpu_mem_used_mb')}MB")

deepseek_df = pd.DataFrame(phi4_records)
deepseek_df['model'] = 'phi'
deepseek_df.to_csv('phi_metrics.csv', index=False)
print("\nSaved deepseek_metrics.csv")
print(deepseek_df[['question_id','output_tokens','latency_sec',
                    'tokens_per_sec','gpu_usage_pct','gpu_temp_c',
                    'gpu_mem_used_mb']].to_string())

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[1/30] done | out_tokens: 435 | latency: 68.09s | tok/s: 6.4 | gpu: 100% | temp: 82C | mem: 10070.19MB
[2/30] done | out_tokens: 192 | latency: 29.26s | tok/s: 6.6 | gpu: 100% | temp: 77C | mem: 10070.19MB
[3/30] done | out_tokens: 359 | latency: 55.29s | tok/s: 6.5 | gpu: 100% | temp: 78C | mem: 10070.19MB
[4/30] done | out_tokens: 166 | latency: 25.57s | tok/s: 6.5 | gpu: 100% | temp: 77C | mem: 10070.19MB
[5/30] done | out_tokens: 165 | latency: 25.35s | tok/s: 6.5 | gpu: 100% | temp: 77C | mem: 10070.19MB
[6/30] done | out_tokens: 172 | latency: 27.39s | tok/s: 6.3 | gpu: 100% | temp: 77C | mem: 10070.19MB
[7/30] done | out_tokens: 222 | latency: 34.14s | tok/s: 6.5 | gpu: 100% | temp: 78C | mem: 10070.19MB
[8/30] done | out_tokens: 139 | latency: 21.43s | tok/s: 6.5 | gpu: 100% | temp: 77C | mem: 10070.19MB
[9/30] done | out_tokens: 448 | latency: 69.07s | tok/s: 6.5 | gpu: 100% | temp: 77C | mem: 10070.19MB
[10/30] done | out_tokens: 389 | latency: 60.08s | tok/s: 6.5 | gpu: 100%

In [11]:
deepseek_df.head()

,question_id,question,answer,prompt_tokens,output_tokens,total_tokens,latency_sec,tokens_per_sec,gpu_usage_pct,gpu_temp_c,gpu_mem_used_mb,model
0,0,A train leaves Station A traveling at 60 mph. ...,"To solve this problem, we need to determine wh...",55,435,490,68.0880,6.39,100,82,10070.19,deepseek
1,1,"If a clock strikes 6 times in 5 seconds, how m...","To solve this problem, we need to understand t...",31,192,223,29.2586,6.56,100,77,10070.19,deepseek
2,2,All prompt engineers are tech enthusiasts. Som...,To determine whether it is definitely true tha...,44,359,403,55.2869,6.49,100,78,10070.19,deepseek
3,3,A bat and a ball cost $1.10 in total. The bat ...,Let's denote the cost of the ball as \( x \). ...,39,166,205,25.5665,6.49,100,77,10070.19,deepseek
4,4,If 5 machines take 5 minutes to make 5 widgets...,"To solve this problem, we first need to unders...",35,165,200,25.3543,6.51,100,77,10070.19,deepseek


In [12]:
llm_groq = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0.4,
    api_key=userdata.get("GROQ_API_KEY"),
    max_retries = 10
)

groq_records = []

for i in range(len(df)):
    start = time.time()
    output = llm_groq.invoke(df["prompt"][i])
    latency = time.time() - start

    # Groq gives token usage directly in metadata
    usage = output.response_metadata.get('token_usage', {})
    prompt_tokens    = usage.get('prompt_tokens', 0)
    output_tokens    = usage.get('completion_tokens', 0)
    total_tokens     = usage.get('total_tokens', 0)
    # use groq's own time if available, else use ours
    groq_latency     = usage.get('completion_time', latency)

    groq_records.append({
        "question_id"      : i,
        "question"         : df["prompt"][i],
        "answer"           : output.content,
        "prompt_tokens"    : prompt_tokens,
        "output_tokens"    : output_tokens,
        "total_tokens"     : total_tokens,
        "latency_sec"      : round(groq_latency, 4),
        "tokens_per_sec"   : round(output_tokens / groq_latency, 2) if groq_latency > 0 else 0,
        "gpu_usage_pct"    : None,   # cloud API — not applicable
        "gpu_temp_c"       : None,
        "gpu_mem_used_mb"  : None,
    })

    print(f"[{i+1}/{len(df)}] done | tokens: {total_tokens} | latency: {groq_latency:.2f}s")

groq_df = pd.DataFrame(groq_records)
groq_df['model'] = 'groq/qwen3-32b'
groq_df.to_csv('groq_metrics.csv', index=False)
print(groq_df.head())

[1/30] done | tokens: 1469 | latency: 3.44s
[2/30] done | tokens: 1878 | latency: 3.95s
[3/30] done | tokens: 1696 | latency: 4.11s
[4/30] done | tokens: 1461 | latency: 3.64s
[5/30] done | tokens: 1884 | latency: 3.38s
[6/30] done | tokens: 2120 | latency: 4.32s
[7/30] done | tokens: 1562 | latency: 3.82s
[8/30] done | tokens: 1078 | latency: 2.98s
[9/30] done | tokens: 1169 | latency: 3.61s
[10/30] done | tokens: 2072 | latency: 4.96s
[11/30] done | tokens: 2080 | latency: 8.68s
[12/30] done | tokens: 768 | latency: 1.52s
[13/30] done | tokens: 1784 | latency: 5.21s
[14/30] done | tokens: 1503 | latency: 2.85s
[15/30] done | tokens: 2077 | latency: 5.63s
[16/30] done | tokens: 1120 | latency: 2.62s
[17/30] done | tokens: 1817 | latency: 7.21s
[18/30] done | tokens: 1936 | latency: 5.97s
[19/30] done | tokens: 784 | latency: 2.32s
[20/30] done | tokens: 2073 | latency: 7.11s
[21/30] done | tokens: 765 | latency: 1.40s
[22/30] done | tokens: 349 | latency: 0.68s
[23/30] done | tokens: 

In [13]:
llm_mistral = ChatMistralAI(
    model="mistral-small-2506",
    temperature=0.4,
    max_tokens=2048,
    api_key=userdata.get("MISTRAL_API_KEY")
)

mistral_records = []

for i in range(len(df)):
    start = time.time()
    output = llm_mistral.invoke(df["prompt"][i])
    latency = time.time() - start

    # Debug on first row
    if i == 0:
        print("usage_metadata:", output.usage_metadata)

    # Same as Groq — usage_metadata is top level on output object
    prompt_tokens = output.usage_metadata.get('input_tokens', 0)
    output_tokens = output.usage_metadata.get('output_tokens', 0)
    total_tokens  = output.usage_metadata.get('total_tokens', 0)

    mistral_records.append({
        "question_id"     : i,
        "question"        : df["prompt"][i],
        "answer"          : output.content,
        "prompt_tokens"   : prompt_tokens,
        "output_tokens"   : output_tokens,
        "total_tokens"    : total_tokens,
        "latency_sec"     : round(latency, 4),
        "tokens_per_sec"  : round(output_tokens / latency, 2) if latency > 0 else 0,
        "gpu_usage_pct"   : None,
        "gpu_temp_c"      : None,
        "gpu_mem_used_mb" : None,
    })

    print(f"[{i+1}/{len(df)}] done | tokens: {total_tokens} | latency: {latency:.2f}s")

mistral_df = pd.DataFrame(mistral_records)
mistral_df['model'] = 'mistral-small-2506'
mistral_df.to_csv('mistral_metrics.csv', index=False)
print(mistral_df.head())

usage_metadata: {'input_tokens': 53, 'output_tokens': 1504, 'total_tokens': 1557}
[1/30] done | tokens: 1557 | latency: 10.42s
[2/30] done | tokens: 900 | latency: 7.52s
[3/30] done | tokens: 1338 | latency: 9.78s
[4/30] done | tokens: 729 | latency: 5.51s
[5/30] done | tokens: 1122 | latency: 7.62s
[6/30] done | tokens: 1119 | latency: 9.23s
[7/30] done | tokens: 353 | latency: 3.29s
[8/30] done | tokens: 168 | latency: 1.59s
[9/30] done | tokens: 1426 | latency: 11.36s
[10/30] done | tokens: 520 | latency: 3.54s
[11/30] done | tokens: 320 | latency: 2.48s
[12/30] done | tokens: 455 | latency: 3.22s
[13/30] done | tokens: 490 | latency: 4.06s
[14/30] done | tokens: 366 | latency: 4.11s
[15/30] done | tokens: 773 | latency: 6.20s
[16/30] done | tokens: 363 | latency: 2.81s
[17/30] done | tokens: 737 | latency: 5.59s
[18/30] done | tokens: 457 | latency: 3.82s
[19/30] done | tokens: 399 | latency: 3.10s
[20/30] done | tokens: 725 | latency: 5.99s
[21/30] done | tokens: 103 | latency: 0.